<a href="https://colab.research.google.com/github/Arfa-Tariq/AstroPlanner-AI/blob/main/notebooks/03_celestial_visibility(v3).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Celestial Visibility Engine (v3) — AstroPlanner AI

This notebook computes and presents which deep-sky objects (from the NGC/IC catalog) and solar system bodies are observable from a given location over an upcoming 7-night window.

## v3 changelog — bug fixes from code audit



### Architecture
- **Skyfield** — true geometric rise/set/transit for planets and Moon, now over a timezone-correct local-noon→noon window
- **Astroplan** — vectorized observability check for NGC candidates, fast batch filtering across thousands of objects
- **Astropy** — coordinate transforms, time systems, and sunrise/sunset


## Setup

In [1]:
!pip install astropy astroplan skyfield requests timezonefinder tzdata -q

import sys, os, json, requests, warnings
import numpy as np
import pandas as pd
from datetime import datetime, timedelta, timezone, date as date_type
from collections import defaultdict
from google.colab import drive

drive.mount('/content/drive')

!git clone https://github.com/Arfa-Tariq/AstroPlanner-AI.git 2>/dev/null || git -C /content/AstroPlanner-AI pull

sys.path.append('/content/AstroPlanner-AI/src')

DATA_DIR = '/content/drive/MyDrive/AstroPlanner/data'
os.makedirs(DATA_DIR, exist_ok=True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 4.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 370.4/370.4 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.0/55.0 MB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 218.1/218.1 kB 15.3 MB/s eta 0:00:00
Mounted at /content/drive


## Imports

In [2]:
# Astropy / Astroplan — NGC catalog observability checks
from astropy.coordinates import EarthLocation, SkyCoord, AltAz
from astropy.time import Time
import astropy.units as u
from astropy.utils.iers import conf as iers_conf
from astroplan import Observer, FixedTarget
from astroplan import AltitudeConstraint, is_observable

# Skyfield — precise planetary rise/set/transit over a timezone-correct window
from skyfield.api import load, wgs84, N, E
from skyfield import almanac

from models import WeeklyPlanRequest, UserProfile

## Suppress warnings + load ephemeris

In [3]:
iers_conf.auto_download = False
iers_conf.auto_max_age  = None

warnings.filterwarnings('ignore', message='.*IERS.*')
warnings.filterwarnings('ignore', message='.*NonRotation.*')
warnings.filterwarnings('ignore', message='.*failed to download.*')
warnings.filterwarnings('ignore', message='.*Angular separation.*')
warnings.filterwarnings('ignore', message='.*unable to download.*')

from skyfield.api import Loader

skyfield_loader = Loader(DATA_DIR)
ts  = skyfield_loader.timescale()

eph_path = os.path.join(DATA_DIR, 'de421.bsp')

try:
    eph = skyfield_loader('de421.bsp')
except ValueError as e:
    if "file starts with b'<!DOCTYP'" in str(e) and os.path.exists(eph_path):
        print(f"Corrupted ephemeris file found at {eph_path}. Deleting and retrying download.")
        os.remove(eph_path)
        eph = skyfield_loader('de421.bsp')
    else:
        raise

print(f"Ephemeris loaded: {eph}")
print(f"Timescale loaded: {ts}")

Ephemeris loaded: Segments from kernel file 'de421.bsp':
  JD 2414864.50 - JD 2471184.50  (1899-07-28 through 2053-10-08)
      0 -> 1    SOLAR SYSTEM BARYCENTER -> MERCURY BARYCENTER
      0 -> 2    SOLAR SYSTEM BARYCENTER -> VENUS BARYCENTER
      0 -> 3    SOLAR SYSTEM BARYCENTER -> EARTH BARYCENTER
      0 -> 4    SOLAR SYSTEM BARYCENTER -> MARS BARYCENTER
      0 -> 5    SOLAR SYSTEM BARYCENTER -> JUPITER BARYCENTER
      0 -> 6    SOLAR SYSTEM BARYCENTER -> SATURN BARYCENTER
      0 -> 7    SOLAR SYSTEM BARYCENTER -> URANUS BARYCENTER
      0 -> 8    SOLAR SYSTEM BARYCENTER -> NEPTUNE BARYCENTER
      0 -> 9    SOLAR SYSTEM BARYCENTER -> PLUTO BARYCENTER
      0 -> 10   SOLAR SYSTEM BARYCENTER -> SUN
      3 -> 301  EARTH BARYCENTER -> MOON
      3 -> 399  EARTH BARYCENTER -> EARTH
      1 -> 199  MERCURY BARYCENTER -> MERCURY
      2 -> 299  VENUS BARYCENTER -> VENUS
      4 -> 499  MARS BARYCENTER -> MARS
Timescale loaded: <skyfield.timelib.Timescale object at 0x791b7bbe4350>


## Load request

In [4]:
with open(f'{DATA_DIR}/current_request.json') as f:
    plan_request = WeeklyPlanRequest.model_validate_json(f.read())

print(f"User     : {plan_request.user.name}")
print(f"Location : {plan_request.user.latitude}\u00b0N, {plan_request.user.longitude}\u00b0E")
print(f"Aperture : {plan_request.user.telescope.aperture_mm}mm")

User     : Andrew
Location : 24.93539°N, 67.11201°E
Aperture : 22.0mm


## Determine observer's local timezone

Every rise/set/transit time computed below comes out of Skyfield/Astropy in UTC. Derived once here from lat/lon (handles DST correctly, unlike a fixed offset) and threaded through every function that produces a displayed time — and, as of v3, through the *search window* itself, not just the display formatting.

In [5]:
from timezonefinder import TimezoneFinder
from zoneinfo import ZoneInfo

def get_observer_timezone(latitude: float, longitude: float) -> ZoneInfo:
    """
    Looks up the IANA timezone for the observing site from its
    coordinates. Falls back to fixed UTC if no timezone is found (e.g.
    international waters) rather than crashing.
    """
    tf = TimezoneFinder()
    tz_name = tf.timezone_at(lat=latitude, lng=longitude)
    if tz_name is None:
        print("  Warning: could not determine timezone from coordinates — falling back to UTC.")
        return ZoneInfo("UTC")
    return ZoneInfo(tz_name)


local_tz = get_observer_timezone(plan_request.user.latitude, plan_request.user.longitude)
print(f"Observer timezone: {local_tz}")

Observer timezone: Asia/Karachi


## Local-noon→noon search window (v3 — replaces UTC-noon→noon)

The old window was `ts.utc(date_str, 12, 0, 0)` through `ts.utc(date_str+1, 12, 0, 0)` — i.e. UTC noon to UTC noon. For any observer west of Greenwich that window ends *before* local noon the next day: a UTC-10 observer's window ran from 2am local through 2am local the next day, clipping dawn twilight and any event that fell after that. Anchoring on the observer's own local noon guarantees the full local night (that evening through the following dawn) sits safely in the middle of the window, regardless of UTC offset.

In [6]:
def get_local_noon_window(date_str: str, local_tz: ZoneInfo):
    """
    Returns (t0, t1) as Skyfield Time objects spanning local noon on
    date_str through local noon the following day, converted to UTC
    internally. Used as the single shared search window for planet
    rise/set/transit — previously these used two independently
    hardcoded UTC windows that could disagree with each other.
    """
    year, month, day = int(date_str[:4]), int(date_str[5:7]), int(date_str[8:10])
    local_noon_start = datetime(year, month, day, 12, 0, 0, tzinfo=local_tz)
    local_noon_end = local_noon_start + timedelta(days=1)
    t0 = ts.from_datetime(local_noon_start.astimezone(timezone.utc))
    t1 = ts.from_datetime(local_noon_end.astimezone(timezone.utc))
    return t0, t1

## Load and prefilter NGC catalog

In [7]:
USEFUL_OBJECT_TYPES = {
    'GX', 'OC', 'GC', 'BN', 'EN', 'RN',
    'PN', 'SNR', 'SC', 'CL+N', 'G+C',
}

def load_ngc_catalog() -> pd.DataFrame:
    """
    Downloads and loads the OpenNGC catalog. Cached to Drive after
    first download so subsequent runs are instant.
    """
    cache_path = f'{DATA_DIR}/ngc_catalog.csv'
    if not os.path.exists(cache_path):
        print("Downloading OpenNGC catalog...")
        url = "https://raw.githubusercontent.com/mattiaverga/OpenNGC/master/database_files/NGC.csv"
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        with open(cache_path, 'w') as f:
            f.write(r.text)
    df = pd.read_csv(cache_path, sep=';', low_memory=False)
    cols = ['Name', 'Type', 'RA', 'Dec', 'V-Mag', 'B-Mag', 'MajAx', 'Common names']
    df = df[[c for c in cols if c in df.columns]]
    df['magnitude'] = pd.to_numeric(df['V-Mag'], errors='coerce')
    df.loc[df['magnitude'].isna(), 'magnitude'] = pd.to_numeric(
        df.loc[df['magnitude'].isna(), 'B-Mag'], errors='coerce'
    )
    print(f"Catalog loaded: {len(df)} total objects")
    return df


def compute_limiting_magnitude(aperture_mm: float) -> float:
    """Standard visual limiting magnitude: 2.1 + 5 * log10(aperture_mm)."""
    return 2.1 + 5 * np.log10(aperture_mm)


def prefilter_catalog(df: pd.DataFrame, user: UserProfile) -> pd.DataFrame:
    """
    Three cheap filters before any sky computation:
    1. Type filter  — keep only meaningful deep-sky object types
    2. Magnitude    — drop objects too faint for this telescope
    3. Declination  — drop objects never reaching >30° at this latitude
    """
    def parse_ra(s):
        try:
            h, m, sec = str(s).split(':')
            return float(h)*15 + float(m)*0.25 + float(sec)*(15/3600)
        except Exception:
            return np.nan

    def parse_dec(s):
        try:
            parts = str(s).split(':')
            sign = -1 if str(s).startswith('-') else 1
            return sign*(abs(float(parts[0])) + float(parts[1])/60 + float(parts[2])/3600)
        except Exception:
            return np.nan

    df = df.copy()
    df['ra_deg']  = df['RA'].apply(parse_ra)
    df['dec_deg'] = df['Dec'].apply(parse_dec)
    df = df.dropna(subset=['ra_deg', 'dec_deg'])

    lim_mag = compute_limiting_magnitude(user.telescope.aperture_mm)
    lat     = user.latitude

    before = len(df)
    df = df[df['Type'].isin(USEFUL_OBJECT_TYPES)]
    print(f"After type filter      : {len(df):5d}  (removed {before-len(df)})")

    before = len(df)
    df = df[df['magnitude'].isna() | (df['magnitude'] <= lim_mag)]
    print(f"After magnitude filter : {len(df):5d}  (removed {before-len(df)})  [limit={lim_mag:.1f}]")

    before = len(df)
    df['max_altitude'] = 90 - abs(lat - df['dec_deg'])
    df = df[df['max_altitude'] >= 30]
    print(f"After declination filter:{len(df):5d}  (removed {before-len(df)})")

    return df.reset_index(drop=True)


ngc_df      = load_ngc_catalog()
filtered_df = prefilter_catalog(ngc_df, plan_request.user)
print(f"\nCandidates for sky computation: {len(filtered_df)}")

Catalog loaded: 13969 total objects
After type filter      :   141  (removed 13821)
After magnitude filter :    19  (removed 122)  [limit=8.8]
After declination filter:   16  (removed 3)

Candidates for sky computation: 16


## Build observer and targets once

In [8]:
def build_observer(user: UserProfile) -> Observer:
    """Builds Astroplan Observer. Created once, reused across all 7 nights."""
    return Observer(
        location=EarthLocation(
            lat=user.latitude  * u.deg,
            lon=user.longitude * u.deg,
            height=0 * u.m
        ),
        name="observer"
    )


def build_target_list(df: pd.DataFrame) -> tuple[list, list]:
    """
    Converts filtered catalog to FixedTarget objects.
    Built once, reused across all 7 nights.
    """
    targets, rows = [], []
    for _, row in df.iterrows():
        try:
            coord = SkyCoord(ra=row['ra_deg']*u.deg, dec=row['dec_deg']*u.deg)
            targets.append(FixedTarget(coord=coord, name=str(row['Name'])))
            rows.append(row)
        except Exception:
            continue
    print(f"Built {len(targets)} FixedTarget objects (reused across all 7 nights)")
    return targets, rows


observer        = build_observer(plan_request.user)
targets, valid_rows = build_target_list(filtered_df)

Built 16 FixedTarget objects (reused across all 7 nights)


## Night window helper (astronomical twilight, unchanged)

In [9]:
def get_night_window(observer: Observer, date_str: str):
    """
    Returns (night_start, night_end) for astronomical twilight.
    Returns (None, None) if no astronomical darkness exists this night
    — relevant for high latitudes (>~48°N) in summer. Unaffected by the
    v3 fixes: Astroplan's twilight_* methods are location-aware and were
    never subject to the noon→noon window bug.
    """
    try:
        midnight = Time(f"{date_str} 23:59:00")
        night_start = observer.twilight_evening_astronomical(midnight, which='nearest')
        night_end   = observer.twilight_morning_astronomical(midnight, which='nearest')
        if (night_end - night_start).to(u.hour).value <= 0:
            return None, None
        return night_start, night_end
    except Exception:
        return None, None

## Skyfield planet rise/set/transit (v3 — fixed pairing, shared window, real sunrise/sunset)

In [10]:
PLANETS = [
    ('moon',                'Moon',    'Moon',   None),
    ('jupiter barycenter',  'Jupiter', 'Planet', -2.9),
    ('saturn barycenter',   'Saturn',  'Planet',  0.7),
    ('mars barycenter',     'Mars',    'Planet',  1.0),
    ('venus barycenter',    'Venus',   'Planet', -4.5),
    ('mercury barycenter',  'Mercury', 'Planet', -0.5),
]


def get_planet_visibility_skyfield(
    user: UserProfile,
    date_str: str,
    local_tz: ZoneInfo,
) -> list[dict]:
    """
    v3: search window is local-noon→noon (get_local_noon_window), shared
    identically between rise/set detection and transit computation.
    Rise/set pairing checks whether the body is already up at window
    start (labeled 'already up', paired with the next set) rather than
    blindly taking the first rise event and first set event independently.
    Daytime visibility uses the observer's actual sunrise/sunset for this
    date instead of a hardcoded UTC window.
    """
    lat = user.latitude
    lon = user.longitude

    t0, t1 = get_local_noon_window(date_str, local_tz)
    window_days = t1 - t0

    skyfield_location = wgs84.latlon(lat * N, lon * E)
    earth             = eph['earth']
    observer_sf       = earth + skyfield_location

    astropy_night_start, astropy_night_end = get_night_window(observer, date_str)

    # Shared time grids, computed once and reused for every planet.
    t_grid = t0 + np.linspace(0, window_days, 145)  # ~10-minute resolution over the full window

    if astropy_night_start is not None:
        t_night_start = ts.from_datetime(astropy_night_start.to_datetime().replace(tzinfo=timezone.utc))
        t_night_end   = ts.from_datetime(astropy_night_end.to_datetime().replace(tzinfo=timezone.utc))
        night_days    = t_night_end - t_night_start
        night_t_grid  = t_night_start + np.linspace(0, night_days, 30)
    else:
        night_t_grid = None

    # Real sunrise/sunset for this date, anchored to local noon so the
    # 'nearest' search reliably lands on THIS day's daytime (not the
    # previous or next day's), unlike an arbitrary UTC anchor.
    try:
        local_noon_astropy = Time(datetime(
            int(date_str[:4]), int(date_str[5:7]), int(date_str[8:10]), 12, 0, 0, tzinfo=local_tz
        ).astimezone(timezone.utc))
        sunrise = observer.sun_rise_time(local_noon_astropy, which='nearest')
        sunset  = observer.sun_set_time(local_noon_astropy, which='nearest')
        t_sunrise   = ts.from_datetime(sunrise.to_datetime().replace(tzinfo=timezone.utc))
        t_sunset    = ts.from_datetime(sunset.to_datetime().replace(tzinfo=timezone.utc))
        day_start, day_end = (t_sunrise, t_sunset) if t_sunrise.tt < t_sunset.tt else (t_sunset, t_sunrise)
        day_days = day_end - day_start
        day_t_grid = day_start + np.linspace(0, day_days, 30)
    except Exception:
        # Fallback: timezone-correct local 06:00-18:00 (NOT raw UTC hours,
        # which would reintroduce the same bug this notebook is fixing).
        year, month, day = int(date_str[:4]), int(date_str[5:7]), int(date_str[8:10])
        local_6am = datetime(year, month, day, 6, 0, 0, tzinfo=local_tz)
        local_6pm = datetime(year, month, day, 18, 0, 0, tzinfo=local_tz)
        t_day_start = ts.from_datetime(local_6am.astimezone(timezone.utc))
        t_day_end   = ts.from_datetime(local_6pm.astimezone(timezone.utc))
        day_t_grid  = t_day_start + np.linspace(0, t_day_end - t_day_start, 25)

    def fmt_local(t):
        return t.utc_datetime().astimezone(local_tz).strftime('%m/%d %H:%M')

    results = []
    for body_key, display_name, obj_type, typical_mag in PLANETS:
        try:
            body = eph[body_key]

            f = almanac.risings_and_settings(
                eph, body, skyfield_location, horizon_degrees=-0.5
            )
            times, events = almanac.find_discrete(t0, t1, f)  # chronologically sorted

            alt0 = observer_sf.at(t0).observe(body).apparent().altaz()[0].degrees
            already_up = alt0 > -0.5

            rise_time = None
            set_time  = None

            if already_up:
                rise_time = "already up"
                for t_ev, ev in zip(times, events):
                    if ev == 0:
                        set_time = fmt_local(t_ev)
                        break
            else:
                for i, (t_ev, ev) in enumerate(zip(times, events)):
                    if ev == 1:
                        rise_time = fmt_local(t_ev)
                        for t_ev2, ev2 in zip(times[i+1:], events[i+1:]):
                            if ev2 == 0:
                                set_time = fmt_local(t_ev2)
                                break
                        break

            if rise_time is not None and set_time is None:
                set_time = "still up at window end"
            elif rise_time is None and set_time is None:
                alt1 = observer_sf.at(t1).observe(body).apparent().altaz()[0].degrees
                if alt0 > -0.5 and alt1 > -0.5:
                    rise_time = set_time = "up all window"
                else:
                    rise_time = set_time = "not visible this window"

            # Transit over the SAME window used for rise/set (previously a
            # separate, independently-buggy UTC 12-36 grid).
            astrometric = observer_sf.at(t_grid).observe(body).apparent()
            alt, az, _  = astrometric.altaz()
            peak_idx    = int(np.argmax(alt.degrees))
            peak_alt    = float(alt.degrees[peak_idx])
            transit_time = fmt_local(t_grid[peak_idx])

            # --- Observable period classification ---
            if night_t_grid is not None:
                night_astro      = observer_sf.at(night_t_grid).observe(body).apparent()
                night_alt, _, _  = night_astro.altaz()
                visible_at_night = float(np.max(night_alt.degrees)) >= 30
            else:
                visible_at_night = False

            day_astro      = observer_sf.at(day_t_grid).observe(body).apparent()
            day_alt, _, _  = day_astro.altaz()
            visible_at_day = float(np.max(day_alt.degrees)) >= 30

            if not visible_at_night and not visible_at_day:
                continue

            if visible_at_night and visible_at_day:
                observable_period = "both"
            elif visible_at_night:
                observable_period = "night"
            else:
                observable_period = "day"

            results.append({
                "name"               : display_name,
                "common_name"        : display_name,
                "type"               : obj_type,
                "magnitude"          : typical_mag,
                "size_arcmin"        : None,
                "ra_deg"             : None,
                "dec_deg"            : None,
                "peak_altitude_deg"  : round(peak_alt, 1),
                "transit_time"       : transit_time,
                "rise_time"          : rise_time,
                "set_time"           : set_time,
                "moon_separation_deg": None,
                "moon_warning"       : False,
                "transits_after_dawn": False,
                "is_solar_system"    : True,
                "observable_period"  : observable_period,
            })

        except Exception:
            continue

    return results

## Nightly visibility computation (v3 — honest field names, finer sampling)

In [11]:
def compute_nightly_visibility(
    observer   : Observer,
    targets    : list,
    valid_rows : list,
    date_str   : str,
    user       : UserProfile,
    local_tz   : ZoneInfo,
) -> list[dict]:
    """
    Computes observable NGC/IC objects for one night.

    v3 field naming: `above_30deg_from_local` / `above_30deg_until_local`
    (previously `rise_time_local` / `set_time_local`) — these represent
    when the object crosses the 30° altitude threshold used for
    observability, NOT its true geometric horizon rise/set. An explicit
    `altitude_threshold_deg` field is included so this is unambiguous
    downstream.
    - 'already above 30° at dusk' = above 30° when astronomical darkness began
    - 'still above 30° at dawn'   = above 30° when astronomical darkness ended
    - HH:MM                       = actually crossed the 30° threshold during the night

    v3 sampling: 6 samples/hour (~10-minute resolution), up from 2/hour
    (~30-minute resolution), matching the planet computation's grid density.

    Moon separation computed per-object at its peak time (not midnight)
    to avoid positional drift error for objects near the 30° threshold.
    Moon separation is stored as a flag, not a hard filter — the
    recommendation engine deprioritizes rather than hard-rejects.

    All NGC objects tagged observable_period='night' since deep-sky
    objects are physically unobservable in daylight regardless of equipment.
    """
    try:
        night_start, night_end = get_night_window(observer, date_str)
        if night_start is None:
            return []

        duration_hours = (night_end - night_start).to(u.hour).value
        n_steps        = max(int(duration_hours * 6), 2)  # v3: ~10-min resolution, was *2 (~30-min)
        time_grid      = night_start + np.linspace(0, duration_hours, n_steps) * u.hour
        time_labels    = [
            t.to_datetime().replace(tzinfo=timezone.utc).astimezone(local_tz).strftime('%H:%M')
            for t in time_grid
        ]

        constraints = [AltitudeConstraint(min=30 * u.deg)]

        observable_mask = is_observable(
            constraints, observer, targets,
            time_range=[night_start, night_end]
        )

        from astropy.coordinates import get_body
        moon_coords    = get_body('moon', time_grid, ephemeris='builtin')
        moon_icrs_grid = SkyCoord(
            ra=moon_coords.ra.deg  * u.deg,
            dec=moon_coords.dec.deg * u.deg,
            frame='icrs'
        )

        visible = []
        for target, row, is_obs in zip(targets, valid_rows, observable_mask):
            if not is_obs:
                continue
            try:
                coord = target.coord
                altaz = coord.transform_to(
                    AltAz(obstime=time_grid, location=observer.location)
                )
                alts = altaz.alt.deg

                peak_idx  = int(np.argmax(alts))
                peak_alt  = float(alts[peak_idx])
                peak_time = time_labels[peak_idx]

                above          = alts >= 30
                rising_indices = np.where(above)[0]
                if len(rising_indices) == 0:
                    continue

                above_from = (
                    "already above 30\u00b0 at dusk"
                    if rising_indices[0] == 0
                    else time_labels[rising_indices[0]]
                )
                above_until = (
                    "still above 30\u00b0 at dawn"
                    if rising_indices[-1] == n_steps - 1
                    else time_labels[rising_indices[-1]]
                )

                transits_after_dawn = (peak_idx == n_steps - 1)

                moon_sep  = float(moon_icrs_grid[peak_idx].separation(coord).deg)
                moon_warn = moon_sep < 30

                mag    = row['magnitude']
                common = str(row.get('Common names', '') or '').split(';')[0].strip() or None

                visible.append({
                    "name"                    : str(row['Name']),
                    "common_name"             : common,
                    "type"                    : str(row.get('Type', 'Unknown')),
                    "magnitude"               : float(mag) if not pd.isna(mag) else None,
                    "size_arcmin"             : float(row['MajAx']) if 'MajAx' in row and not pd.isna(row.get('MajAx')) else None,
                    "ra_deg"                  : round(float(row['ra_deg']), 4),
                    "dec_deg"                 : round(float(row['dec_deg']), 4),
                    "peak_altitude_deg"       : round(peak_alt, 1),
                    "peak_time_local"         : peak_time,
                    "altitude_threshold_deg"  : 30,
                    "above_30deg_from_local"  : above_from,
                    "above_30deg_until_local" : above_until,
                    "transits_after_dawn"     : transits_after_dawn,
                    "moon_separation_deg"     : round(moon_sep, 1),
                    "moon_warning"            : moon_warn,
                    "is_solar_system"         : False,
                    "observable_period"       : "night",
                })

            except Exception:
                continue

        for obj in visible:
            b = max(0, (15 - (obj['magnitude'] or 15)) / 15)
            a = obj['peak_altitude_deg'] / 90
            obj['_score'] = 0.5 * a + 0.5 * b

        visible.sort(key=lambda x: x['_score'], reverse=True)

        TYPE_CAPS = {
            'GX'  : 30, 'OC': 15, 'GC': 15,
            'BN'  : 8,  'EN': 8,  'RN': 5,
            'PN'  : 10, 'SNR': 5, 'SC': 3,
            'CL+N': 5,  'G+C': 3,
        }
        counts  = defaultdict(int)
        diverse = []
        for obj in visible:
            t = obj['type']
            if counts[t] < TYPE_CAPS.get(t, 5):
                diverse.append(obj)
                counts[t] += 1
            if len(diverse) >= 100:
                break

        for obj in diverse:
            obj.pop('_score', None)

        planets = get_planet_visibility_skyfield(user, date_str, local_tz) or []
        return planets + diverse

    except Exception as e:
        print(f"\n  Warning: visibility computation failed for {date_str}: {e}")
        return []

## Weekly loop

In [12]:
def get_weekly_visibility(
    user       : UserProfile,
    observer   : Observer,
    targets    : list,
    valid_rows : list,
    start_date : date_type,
    local_tz   : ZoneInfo,
) -> list[dict]:
    """
    Computes visibility for all 7 nights, anchored to start_date
    (plan_request.generated_at, fixed once in notebook 01) so notebooks
    02/03 always agree on the 7-day window even if run on different
    calendar days.
    """
    weekly = []

    for offset in range(7):
        date_str = (start_date + timedelta(days=offset)).strftime('%Y-%m-%d')
        print(f"Computing {date_str}...", end=" ")

        try:
            nightly = compute_nightly_visibility(
                observer, targets, valid_rows, date_str, user, local_tz
            )
        except Exception as e:
            print(f"ERROR: {e}")
            nightly = []

        if nightly is None:
            nightly = []

        n_planets = sum(1 for o in nightly if o.get('is_solar_system'))
        n_dso     = sum(1 for o in nightly if not o.get('is_solar_system'))
        print(
            f"{len(nightly)} objects  "
            f"({n_planets} planets/moon + {n_dso} deep-sky)"
        )

        weekly.append({
            "date"                : date_str,
            "day_offset"          : offset,
            "timezone"            : str(local_tz),
            "visible_object_count": len(nightly),
            "objects"             : nightly,
        })

    return weekly

## Run and save

In [13]:
weekly_visibility = get_weekly_visibility(
    plan_request.user, observer, targets, valid_rows, plan_request.generated_at, local_tz
)

with open(f'{DATA_DIR}/weekly_visibility.json', 'w') as f:
    json.dump(weekly_visibility, f, indent=2, default=str)

print(f"\nSaved to {DATA_DIR}/weekly_visibility.json")
print("\n=== Visibility Summary ===")
for night in weekly_visibility:
    print(f"{night['date']}: {night['visible_object_count']} total objects")

Computing 2026-07-27... 19 objects  (6 planets/moon + 13 deep-sky)
Computing 2026-07-28... 19 objects  (6 planets/moon + 13 deep-sky)
Computing 2026-07-29... 19 objects  (6 planets/moon + 13 deep-sky)
Computing 2026-07-30... 19 objects  (6 planets/moon + 13 deep-sky)
Computing 2026-07-31... 19 objects  (6 planets/moon + 13 deep-sky)
Computing 2026-08-01... 19 objects  (6 planets/moon + 13 deep-sky)
Computing 2026-08-02... 19 objects  (6 planets/moon + 13 deep-sky)

Saved to /content/drive/MyDrive/AstroPlanner/data/weekly_visibility.json

=== Visibility Summary ===
2026-07-27: 19 total objects
2026-07-28: 19 total objects
2026-07-29: 19 total objects
2026-07-30: 19 total objects
2026-07-31: 19 total objects
2026-08-01: 19 total objects
2026-08-02: 19 total objects


## Spot check and testing of outputs

In [14]:
best = max(weekly_visibility, key=lambda n: n['visible_object_count'])
print(f"Best night: {best['date']}  ({best['visible_object_count']} objects, times shown in {best['timezone']})\n")

print("=== Solar System Bodies (true geometric times) ===")
for obj in best['objects']:
    if obj.get('is_solar_system'):
        period = obj.get('observable_period', 'night')
        period_label = {
            'night': '🌙 night only',
            'day'  : '☀️  day only',
            'both' : '🌓 day + night',
        }.get(period, period)
        print(
            f"  {obj['name']:8}  "
            f"peak={obj['peak_altitude_deg']}\u00b0 at {obj['transit_time']}  "
            f"rises={obj['rise_time']}  sets={obj['set_time']}  "
            f"[{period_label}]"
        )

print("\n=== Top 15 Deep-Sky Objects ===")
dso = [o for o in best['objects'] if not o.get('is_solar_system')]
for obj in dso[:15]:
    moon_flag = "  ⚠ near moon" if obj.get('moon_warning') else ""
    dawn_flag = "  (transits after dawn)" if obj.get('transits_after_dawn') else ""
    print(
        f"  {obj['name']:10} {(obj['common_name'] or ''):25} "
        f"type={obj['type']:5} mag={str(obj['magnitude']):5} "
        f"peak={obj['peak_altitude_deg']}\u00b0 at {obj['peak_time_local']}  "
        f"above30\u00b0 from={obj['above_30deg_from_local']}  until={obj['above_30deg_until_local']}"
        f"{moon_flag}{dawn_flag}"
    )

print("\n=== Object Type Distribution ===")
dist = defaultdict(int)
for o in best['objects']:
    dist[o['type']] += 1
for t, c in sorted(dist.items(), key=lambda x: -x[1]):
    print(f"  {t:10}: {c}")

print("\n=== Observable Period Breakdown ===")
for period, label in [('night', '🌙 night'), ('day', '☀️  day'), ('both', '🌓 both')]:
    count = sum(1 for o in best['objects'] if o.get('observable_period') == period)
    if count:
        print(f"  {label}: {count} objects")

Best night: 2026-07-27  (19 objects, times shown in Asia/Karachi)

=== Solar System Bodies (true geometric times) ===
  Moon      peak=37.9° at 07/27 23:10  rises=07/27 17:58  sets=07/28 04:29  [🌙 night only]
  Jupiter   peak=84.2° at 07/27 12:50  rises=already up  sets=07/27 19:24  [☀️  day only]
  Saturn    peak=68.6° at 07/28 05:10  rises=07/27 22:59  sets=07/28 11:14  [🌓 day + night]
  Mars      peak=88.0° at 07/28 09:30  rises=already up  sets=07/27 16:15  [☀️  day only]
  Venus     peak=69.6° at 07/27 15:30  rises=already up  sets=07/27 21:42  [☀️  day only]
  Mercury   peak=84.3° at 07/28 11:20  rises=already up  sets=07/27 18:01  [☀️  day only]

=== Top 15 Deep-Sky Objects ===
  NGC6853    Dumbbell Nebula           type=PN    mag=7.4   peak=87.8° at 00:12  above30° from=already above 30° at dusk  until=still above 30° at dawn
  NGC6960    Veil Nebula,Filamentary Nebula,Western Veil type=SNR   mag=7.0   peak=84.2° at 00:53  above30° from=already above 30° at dusk  until=still ab

In [15]:
# --- Full 7-night summary (all nights, not just the best one) ---
print("=== Weekly Visibility Overview ===\n")

for night in weekly_visibility:
    n_planets = sum(1 for o in night['objects'] if o.get('is_solar_system'))
    n_dso     = sum(1 for o in night['objects'] if not o.get('is_solar_system'))
    marker    = "  ← best" if night is best else ""

    print(
        f"{night['date']}  "
        f"({night['visible_object_count']:2d} objects: "
        f"{n_planets} planets/moon + {n_dso} deep-sky)"
        f"{marker}"
    )

    for obj in night['objects']:
        if obj.get('is_solar_system'):
            print(
                f"    {obj['name']:8}  "
                f"peak={obj['peak_altitude_deg']}\u00b0 at {obj['transit_time']}  "
                f"rises={obj['rise_time']}  sets={obj['set_time']}  "
                f"[{obj.get('observable_period', 'night')}]"
            )

    print(f"    ...+{n_dso} deep-sky objects (see 'Spot check' cell above for full detail on the best night)\n")

=== Weekly Visibility Overview ===

2026-07-27  (19 objects: 6 planets/moon + 13 deep-sky)  ← best
    Moon      peak=37.9° at 07/27 23:10  rises=07/27 17:58  sets=07/28 04:29  [night]
    Jupiter   peak=84.2° at 07/27 12:50  rises=already up  sets=07/27 19:24  [day]
    Saturn    peak=68.6° at 07/28 05:10  rises=07/27 22:59  sets=07/28 11:14  [both]
    Mars      peak=88.0° at 07/28 09:30  rises=already up  sets=07/27 16:15  [day]
    Venus     peak=69.6° at 07/27 15:30  rises=already up  sets=07/27 21:42  [day]
    Mercury   peak=84.3° at 07/28 11:20  rises=already up  sets=07/27 18:01  [day]
    ...+13 deep-sky objects (see 'Spot check' cell above for full detail on the best night)

2026-07-28  (19 objects: 6 planets/moon + 13 deep-sky)
    Moon      peak=40.7° at 07/29 00:00  rises=07/28 18:42  sets=07/29 05:25  [night]
    Jupiter   peak=84.2° at 07/28 12:40  rises=already up  sets=07/28 19:20  [day]
    Saturn    peak=68.5° at 07/29 05:00  rises=07/28 22:55  sets=07/29 11:10  [bo